# Acoustic PDM — Phase 3: Notebook 04
## Model Training, Baselines & Dual-Stage Deep Hybrid Architecture

**Project:** Acoustic Predictive Maintenance (Acoustic PDM)  
**Dataset:** Hitachi MIMII — 4 industrial machine types (**Fan, Pump, Slider, Valve**) at 6 dB SNR, Machine ID 00  
**Input:** Preprocessed tensor blocks from **Notebook 03** (`train_normal.npy`, `val_normal.npy`, etc.)  
**Runtime:** GPU Recommended (Kaggle T4 x 2 or P100)  

---

### What is this notebook about?

In Phase 2 (Notebook 03), we converted raw acoustic soundwaves into 1.13+ million normalized spectrogram patches of shape `(1, 128, 5)`.  
In this notebook, we train our complete suite of deep learning autoencoders, benchmark baselines, and the state-of-the-art **Dual-Stage Deep Hybrid Model**.

### Model Architectures Trained in this Notebook:

1. **Conv2D Autoencoder (Conv2D-AE) [Primary SOTA Backbone]:**
   - 3-layer Convolutional Encoder with LeakyReLU and batch normalization down to a 32-dimensional bottleneck $z \in \mathbb{R}^{32}$.
   - 3-layer Transposed Convolutional Decoder reconstructing original frequency-time context `(1, 128, 5)`.
2. **Fully-Connected Autoencoder (FC-AE) [Deep Baseline]:**
   - Multi-layer dense bottleneck autoencoder on flattened 640-dimensional vectors.
3. **Recurrent LSTM Autoencoder (LSTM-AE) [Temporal Baseline]:**
   - 2-layer LSTM modeling temporal frame transitions across 5 context windows.
4. **Shallow Machine Learning Baselines:**
   - **Isolation Forest (IF)** on handcrafted DSP representations.
   - **One-Class Support Vector Machine (OC-SVM)** with RBF kernel.
   - **XGBoost Classifier** (Supervised baseline demonstrating the *open-world failure trap*).
5. **Dual-Stage Deep Hybrid Model (Conv2D-AE + Latent Isolation Forest):**
   - Fuses physical reconstruction error $S_{\text{recon}}$ with latent manifold density $S_{\text{latent}}$:
     $$S_{\text{final}} = 0.6 \cdot \tilde{S}_{\text{recon}} + 0.4 \cdot \tilde{S}_{\text{latent}}$$

---
### Step 0: Environment Bootstrap & Input Discovery

This cell sets random seeds for 100% reproducibility (`seed=42`), detects GPU acceleration, and locates the preprocessed `.npy` tensors produced by Notebook 03.

> **Kaggle Adaptation:** Training hyperparameters (batch size, epochs, patience) are tuned for optimal Kaggle T4 GPU runtime. Canonical defaults live in `configs/config.yaml`; see Step 5 markdown for the specific deviations and rationale.

In [ ]:
import os
import copy
import gc
import random
from pathlib import Path
import numpy as np
import torch

# ── Reproducibility Seeds ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# ── Path Discovery ──
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
    # Search for NB03 preprocessed output folder across /kaggle/input
    _data_dir = None
    for _root, _dirs, _ in os.walk("/kaggle/input"):
        if "processed" in _dirs:
            _data_dir = Path(_root) / "processed"
            break
        elif "fan" in _dirs and "pump" in _dirs:
            _data_dir = Path(_root)
            break
    if _data_dir is None:
        # Check if run locally in same session
        _data_dir = Path("/kaggle/working/data/processed")
    PROCESSED_DIR = _data_dir
else:
    _cwd = Path(os.getcwd()).resolve()
    PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

MODELS_DIR  = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFIGS_DIR = PROJECT_ROOT / "configs"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Device Configuration ──
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print(f"\U0001f680 ACOUSTIC PDM \u2014 PHASE 3 TRAINING ENGINE")
print(f"Environment:    {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Device:         {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Processed Data: {PROCESSED_DIR}")
print(f"Models Output:  {MODELS_DIR}")
print("=" * 70)

---
### Step 1: Import Libraries

Import standard deep learning and scientific computing frameworks (PyTorch, scikit-learn, XGBoost, matplotlib, joblib).

In [ ]:
import time
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
import xgboost as xgb

print("✓ Libraries successfully imported.")

---
### Step 2: PyTorch Dataset & DataLoader

The `AcousticTensorDataset` wraps `(N, 1, 128, 5)` `.npy` arrays into memory-efficient PyTorch tensors.

In [ ]:
class AcousticTensorDataset(Dataset):
    def __init__(self, data_array):
        if isinstance(data_array, (str, Path)):
            self.data = np.load(str(data_array)).astype(np.float32)
        else:
            self.data = data_array.astype(np.float32)
            
        if self.data.ndim == 3:
            self.data = np.expand_dims(self.data, axis=1)
            
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx])

def load_machine_tensors(machine_name, base_dir=PROCESSED_DIR):
    """Load train, validation, and test tensors for a specific machine."""
    m_dir = Path(base_dir) / machine_name
    train_normal = np.load(m_dir / "train_normal.npy").astype(np.float32)
    val_normal   = np.load(m_dir / "val_normal.npy").astype(np.float32)
    test_normal  = np.load(m_dir / "test_normal.npy").astype(np.float32)
    test_anomaly = np.load(m_dir / "test_anomaly.npy").astype(np.float32)
    return train_normal, val_normal, test_normal, test_anomaly

# Verify loading on first machine (Fan)
sample_train, sample_val, _, _ = load_machine_tensors("fan")
print(f"✓ Fan Tensors Loaded: Train Normal: {sample_train.shape} | Val Normal: {sample_val.shape}")

---
### Step 3: Neural Network Architectures

Here we define our deep learning models:
1. **`Conv2DAutoencoder`**: 2D Convolutions capture joint time-frequency patterns across $(128, 5)$ blocks.
2. **`FCAutoencoder`**: Dense Multi-Layer Perceptron baseline.
3. **`LSTMAutoencoder`**: Temporal recurrence baseline.

In [ ]:
class Conv2DAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder: (1, 128, 5) -> (32, 64, 3) -> (64, 32, 2) -> (128, 16, 1) -> 32
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.encoder_fc = nn.Linear(128 * 16 * 1, latent_dim)
        
        # Decoder: 32 -> 2048 -> (128, 16, 1) -> (64, 32, 2) -> (32, 64, 3) -> (1, 128, 5)
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128 * 16 * 1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=(1, 1)),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),
        )
        
    def encode(self, x):
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)
        return self.encoder_fc(h)
        
    def decode(self, z):
        h = self.decoder_fc(z)
        h = h.view(-1, 128, 16, 1)
        return self.decoder_conv(h)
        
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)


class FCAutoencoder(nn.Module):
    def __init__(self, input_dim=640, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Linear(512, input_dim)
        )
        
    def encode(self, x):
        return self.encoder(x.view(x.size(0), -1))
        
    def decode(self, z):
        return self.decoder(z).view(-1, 1, 128, 5)
        
    def forward(self, x):
        return self.decode(self.encode(x))


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_mels=128, context_frames=5, hidden_dim=64, latent_dim=32):
        super().__init__()
        self.context_frames = context_frames
        self.encoder_lstm = nn.LSTM(input_size=n_mels, hidden_size=hidden_dim, num_layers=2, batch_first=True)
        self.encoder_fc   = nn.Linear(hidden_dim, latent_dim)
        self.decoder_fc   = nn.Linear(latent_dim, hidden_dim)
        self.decoder_lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=2, batch_first=True)
        self.output_fc   = nn.Linear(hidden_dim, n_mels)
        
    def encode(self, x):
        seq = x.squeeze(1).permute(0, 2, 1)  # (B, 5, 128)
        _, (h_n, _) = self.encoder_lstm(seq)
        return self.encoder_fc(h_n[-1])
        
    def decode(self, z):
        h = self.decoder_fc(z).unsqueeze(1).repeat(1, self.context_frames, 1)
        out, _ = self.decoder_lstm(h)
        recon_seq = self.output_fc(out)
        return recon_seq.permute(0, 2, 1).unsqueeze(1)
        
    def forward(self, x):
        return self.decode(self.encode(x))

# Sanity check forward pass
dummy = torch.randn(8, 1, 128, 5)
m_conv = Conv2DAutoencoder()
assert m_conv(dummy).shape == dummy.shape, f"Shape mismatch: {m_conv(dummy).shape}"
print(f"✓ Neural architectures initialized. Conv2D-AE Parameters: {sum(p.numel() for p in m_conv.parameters()):,}")

---
### Step 4: Training Engine Routine

Encapsulates PyTorch optimization loop with MSE loss, Adam optimizer, `ReduceLROnPlateau`, and early stopping.

> **Implementation Note:** Best model weights are preserved using `copy.deepcopy(model.state_dict())` to prevent in-place parameter corruption. A shallow `.copy()` on `state_dict()` shares tensor storage with the live model — the optimizer silently overwrites the "best" checkpoint during subsequent epochs.

In [ ]:
def train_model(
    model,
    train_data,
    val_data,
    batch_size=64,
    epochs=35,
    lr=0.001,
    patience=8,
    device=DEVICE
):
    model = model.to(device)
    train_loader = DataLoader(AcousticTensorDataset(train_data), batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader   = DataLoader(AcousticTensorDataset(val_data), batch_size=batch_size, shuffle=False, drop_last=False)
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    
    history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    best_state = None
    patience_cnt = 0
    
    for epoch in range(1, epochs + 1):
        model.train()
        t_loss = 0.0
        t_cnt = 0
        for x in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            recon = model(x)
            loss = criterion(recon, x)
            loss.backward()
            optimizer.step()
            t_loss += loss.item() * x.size(0)
            t_cnt += x.size(0)
            
        train_loss = t_loss / t_cnt
        
        model.eval()
        v_loss = 0.0
        v_cnt = 0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(device)
                recon = model(x)
                loss = criterion(recon, x)
                v_loss += loss.item() * x.size(0)
                v_cnt += x.size(0)
                
        val_loss = v_loss / v_cnt
        scheduler.step(val_loss)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch [{epoch:02d}/{epochs:02d}] \u2014 Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"  \u23f9 Early stopped at epoch {epoch} (Best Val Loss: {best_val_loss:.6f})")
                break
                
    if best_state is not None:
        model.load_state_dict(best_state)
        
    return model, history

---
### Step 5: Main Training Loop Across All 4 Machines

We train our primary **Conv2D-AE** models across the 4 industrial machine types (**Fan**, **Pump**, **Slider**, **Valve** at `id_00`).

> **Kaggle Runtime Adaptations** (vs. `configs/config.yaml` defaults):  
> `batch_size=128` (config: 32), `epochs=40` (config: 50), `patience=8` (config: 10).  
> These values are tuned to complete 4-machine training within the Kaggle T4 GPU session time limit while maintaining convergence quality.

In [ ]:
MACHINES = ["fan", "pump", "slider", "valve"]
all_histories = {}
trained_conv_models = {}

for machine in MACHINES:
    print(f"\n{'='*70}")
    print(f"  TRAINING PRIMARY Conv2D-AE: {machine.upper()}")
    print(f"{'='*70}")
    
    train_normal, val_normal, _, _ = load_machine_tensors(machine)
    
    model = Conv2DAutoencoder(latent_dim=32)
    start_t = time.time()
    trained_model, hist = train_model(
        model=model,
        train_data=train_normal,
        val_data=val_normal,
        batch_size=128,
        epochs=40,
        lr=0.001,
        patience=8
    )
    elapsed = time.time() - start_t
    print(f"\u2713 Completed {machine.upper()} in {elapsed:.1f}s | Min Val Loss: {min(hist['val_loss']):.6f}")
    
    # Save model weights
    save_path = MODELS_DIR / f"best_conv2d_ae_{machine}.pth"
    torch.save(trained_model.state_dict(), str(save_path))
    print(f"  Saved: {save_path.name}")
    
    all_histories[machine] = hist
    trained_conv_models[machine] = trained_model
    
    # Memory cleanup for Kaggle RAM ceiling
    del train_normal, val_normal
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

---
### Step 6: Visualizing Training & Validation Loss Curves

Plot the learning dynamics for all 4 machines to verify convergence and zero overfitting.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, machine in enumerate(MACHINES):
    hist = all_histories[machine]
    ax = axes[idx]
    epochs_range = range(1, len(hist["train_loss"]) + 1)
    
    ax.plot(epochs_range, hist["train_loss"], label="Train Loss (MSE)", color="#1f77b4", lw=2)
    ax.plot(epochs_range, hist["val_loss"], label="Val Loss (MSE)", color="#ff7f0e", lw=2, linestyle="--")
    ax.set_title(f"Conv2D-AE Convergence: {machine.upper()}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epochs")
    ax.set_ylabel("MSE Reconstruction Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
curve_path = REPORTS_DIR / "conv2d_ae_training_curves.png"
plt.savefig(curve_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"✓ Saved loss curves to: {curve_path.name}")

---
### Step 7: Benchmark Baseline Models

Here we train the comparative baselines for the Fan asset:
1. **FC-AE** (Dense Neural Network Autoencoder)
2. **LSTM-AE** (Recurrent Sequence Autoencoder)
3. **Shallow Isolation Forest**
4. **One-Class SVM**
5. **XGBoost Classifier** (Trained on normal + subset anomaly to demonstrate open-world failure)

In [ ]:
print(f"\n{'='*70}")
print(f"  TRAINING COMPARATIVE BASELINES (FAN ASSET)")
print(f"{'='*70}")

train_fan, val_fan, test_fan_norm, test_fan_anom = load_machine_tensors("fan")

# 1. FC-AE
print("1. Training FC-AE...")
fc_model = FCAutoencoder(input_dim=640, latent_dim=32)
fc_model, _ = train_model(fc_model, train_fan, val_fan, batch_size=128, epochs=25, lr=0.001)
torch.save(fc_model.state_dict(), str(MODELS_DIR / "best_fc_ae_fan.pth"))

# 2. LSTM-AE (Subsampled for faster training)
print("2. Training LSTM-AE...")
lstm_model = LSTMAutoencoder(n_mels=128, context_frames=5, hidden_dim=64, latent_dim=32)
lstm_model, _ = train_model(lstm_model, train_fan[:50000], val_fan[:5000], batch_size=128, epochs=15, lr=0.001)
torch.save(lstm_model.state_dict(), str(MODELS_DIR / "best_lstm_ae_fan.pth"))

# 3. Shallow Machine Learning Baselines
print("3. Training Shallow Baselines (Isolation Forest, One-Class SVM, XGBoost)...")
flat_train = train_fan.reshape(len(train_fan), -1)
sub_idx = np.random.choice(len(flat_train), size=25000, replace=False)
flat_sub = flat_train[sub_idx]

# Isolation Forest
if_shallow = IsolationForest(n_estimators=100, contamination=0.05, random_state=SEED, n_jobs=-1)
if_shallow.fit(flat_sub)
joblib.dump(if_shallow, str(MODELS_DIR / "shallow_iforest_fan.joblib"))

# One-Class SVM
ocsvm = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
ocsvm.fit(flat_sub[:5000])
joblib.dump(ocsvm, str(MODELS_DIR / "shallow_ocsvm_fan.joblib"))

# XGBoost Supervised Baseline (dynamic label sizing to prevent data-label mismatch)
flat_anom = test_fan_anom.reshape(len(test_fan_anom), -1)[:5000]
n_anom = len(flat_anom)
X_sup = np.vstack([flat_sub[:n_anom], flat_anom])
y_sup = np.array([0] * n_anom + [1] * n_anom)
xgb_clf = xgb.XGBClassifier(max_depth=6, n_estimators=100, learning_rate=0.1, random_state=SEED, eval_metric="logloss")
xgb_clf.fit(X_sup, y_sup)
joblib.dump(xgb_clf, str(MODELS_DIR / "supervised_xgb_fan.joblib"))

print("\u2713 All baseline models trained and saved to models/ successfully.")

---
### Step 8: Training Dual-Stage Deep Hybrid Models (Conv2D-AE + Latent IF)

In this step, we construct our **Dual-Stage Deep Hybrid Architecture**:
1. Pass normal training tensors through the trained Conv2D-AE encoder to extract 32-dimensional bottleneck latent vectors $z \in \mathbb{R}^{32}$.
2. Fit an `IsolationForest` on this compressed latent manifold.
3. Save the fitted `IsolationForest` for all 4 machines.

In [ ]:
def extract_latents(model, data_array, batch_size=256, device=DEVICE):
    model = model.to(device)
    model.eval()
    loader = DataLoader(AcousticTensorDataset(data_array), batch_size=batch_size, shuffle=False)
    latents = []
    with torch.no_grad():
        for x in loader:
            z = model.encode(x.to(device))
            latents.append(z.cpu().numpy())
    return np.concatenate(latents, axis=0)

hybrid_if_models = {}

for machine in MACHINES:
    print(f"\nFitting Stage 2 Latent Isolation Forest: {machine.upper()}...")
    train_normal, _, _, _ = load_machine_tensors(machine)
    ae_model = trained_conv_models[machine]
    
    # Extract latent vectors
    z_train = extract_latents(ae_model, train_normal)
    
    # Subsample if large for fast, high-density fitting
    fit_z = z_train[np.random.choice(len(z_train), size=min(len(z_train), 50000), replace=False)]
    
    # Fit Latent Isolation Forest
    if_latent = IsolationForest(n_estimators=100, contamination=0.05, random_state=SEED, n_jobs=-1)
    if_latent.fit(fit_z)
    
    # Save hybrid checkpoint
    joblib_path = MODELS_DIR / f"hybrid_latent_if_{machine}.joblib"
    joblib.dump(if_latent, str(joblib_path))
    print(f"\u2713 Stage 2 Latent IF fitted & saved: {joblib_path.name}")
    hybrid_if_models[machine] = if_latent
    
    # Memory cleanup
    del train_normal, z_train, fit_z
    gc.collect()

---
### Step 9: Latent Space 2D Projection (PCA Analysis)

Visualize the 32-dimensional latent representations of Normal vs. Anomaly soundscapes in 2D space to verify manifold separation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, machine in enumerate(["fan", "valve"]):
    ae_model = trained_conv_models[machine]
    _, _, test_norm, test_anom = load_machine_tensors(machine)
    
    z_norm = extract_latents(ae_model, test_norm[:3000])
    z_anom = extract_latents(ae_model, test_anom[:3000])
    
    z_all = np.vstack([z_norm, z_anom])
    pca = PCA(n_components=2, random_state=SEED)
    z_2d = pca.fit_transform(z_all)
    
    ax = axes[idx]
    ax.scatter(z_2d[:len(z_norm), 0], z_2d[:len(z_norm), 1], c="#2ca02c", alpha=0.5, label="Normal (Healthy)", s=15)
    ax.scatter(z_2d[len(z_norm):, 0], z_2d[len(z_norm):, 1], c="#d62728", alpha=0.5, label="Anomaly (Faulty)", s=15)
    ax.set_title(f"Latent Manifold Projection (PCA): {machine.upper()}", fontsize=12, fontweight="bold")
    ax.set_xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
    ax.set_ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
pca_path = REPORTS_DIR / "latent_space_pca.png"
plt.savefig(pca_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"✓ Saved latent projection to: {pca_path.name}")

---
### Step 10: Model Checkpoint Verification

Audit all exported `.pth` and `.joblib` model checkpoint artifacts in `models/`.

In [ ]:
print("=" * 75)
print("🔍 MODEL CHECKPOINT AUDIT SUMMARY")
print("=" * 75)

checkpoints = list(MODELS_DIR.glob("*"))
audit_data = []

for cp in sorted(checkpoints):
    size_mb = cp.stat().st_size / (1024 * 1024)
    audit_data.append({
        "Model Checkpoint": cp.name,
        "Type": cp.suffix,
        "Size (MB)": f"{size_mb:.2f} MB"
    })

df_models = pd.DataFrame(audit_data)
print(df_models.to_string(index=False))
print("=" * 75)
print("🎉 PHASE 3 COMPLETE: All models trained and ready for NB05 Comparative Evaluation!")
print("=" * 75)